## Credit Portfolio Risk Analytics Framework - Expected Loss Estimation & Stress Testing - Data Preprocessing
* This file has all the code used to preprocess the data for credit risk modeling. 
* for main model, please refer to the "**Credit Portfolio Risk Analytics Framework - Expected Loss Estimation & Stress Testing.ipynb**" file.

In [2]:
# Libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Importing CSV
og_data = pd.read_csv("corporate_rating.csv")

In [4]:
# Creating a copy for cross checking
data = og_data.copy()

In [5]:
# Adjusting display
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.options.display.max_columns = None


In [6]:
data.columns

Index(['Rating', 'Name', 'Symbol', 'Rating Agency Name', 'Date', 'Sector',
       'currentRatio', 'quickRatio', 'cashRatio', 'daysOfSalesOutstanding',
       'netProfitMargin', 'pretaxProfitMargin', 'grossProfitMargin',
       'operatingProfitMargin', 'returnOnAssets', 'returnOnCapitalEmployed',
       'returnOnEquity', 'assetTurnover', 'fixedAssetTurnover',
       'debtEquityRatio', 'debtRatio', 'effectiveTaxRate',
       'freeCashFlowOperatingCashFlowRatio', 'freeCashFlowPerShare',
       'cashPerShare', 'companyEquityMultiplier', 'ebitPerRevenue',
       'enterpriseValueMultiple', 'operatingCashFlowPerShare',
       'operatingCashFlowSalesRatio', 'payablesTurnover'],
      dtype='object')

In [7]:
# Convert date to datetime
data['Date'] = pd.to_datetime(data['Date'])

# Extract year from date
data['year'] = data['Date'].dt.year

In [8]:
data['Rating'].unique()

array(['A', 'BBB', 'AA', 'BB', 'B', 'CCC', 'D', 'CC', 'AAA', 'C'],
      dtype=object)

In [9]:
# mapping rating to score
rating_map = {
    'AAA': 10,
    'AA': 9,
    'A': 8,
    'BBB': 7,
    'BB': 6,
    'B': 5,
    'CCC': 4,
    'C': 3,
    'CC': 2,
    'D': 1
}

data['Rating_score'] = data['Rating'].map(rating_map)

In [10]:
# Selecting Low conservative score to assess risk of repeating score for the company in same year

data = data.loc[
    data.groupby(['Symbol','year'])['Rating_score'].idxmin().reset_index(drop=True)
]

In [11]:
data[data['Name'] == 'Whirlpool Corporation']

,Rating,Name,Symbol,Rating Agency Name,Date,Sector,currentRatio,quickRatio,cashRatio,daysOfSalesOutstanding,netProfitMargin,pretaxProfitMargin,grossProfitMargin,operatingProfitMargin,returnOnAssets,returnOnCapitalEmployed,returnOnEquity,assetTurnover,fixedAssetTurnover,debtEquityRatio,debtRatio,effectiveTaxRate,freeCashFlowOperatingCashFlowRatio,freeCashFlowPerShare,cashPerShare,companyEquityMultiplier,ebitPerRevenue,enterpriseValueMultiple,operatingCashFlowPerShare,operatingCashFlowSalesRatio,payablesTurnover,year,Rating_score
3,BBB,Whirlpool Corporation,WHR,Fitch Ratings,2012-06-15,Consumer Durables,1.019851,0.510402,0.176116,41.161738,0.020894,-0.012858,0.138059,0.042430,0.025690,-0.027015,0.093279,1.229563,6.017408,2.630950,0.724590,1.816667,-0.147170,-1.015625,14.440104,3.630950,-0.012858,4.080741,6.901042,0.028394,4.581150,2012,7
1,BBB,Whirlpool Corporation,WHR,Egan-Jones Ratings Company,2014-02-13,Consumer Durables,1.033559,0.498234,0.203120,38.991156,0.044062,0.048857,0.175715,0.066546,0.053204,0.104800,0.167953,1.207476,6.171983,2.156783,0.683222,0.074155,0.541997,8.625473,17.402270,3.156783,0.048857,6.460618,15.914250,0.067239,4.002846,2014,7
2,BBB,Whirlpool Corporation,WHR,Fitch Ratings,2015-03-06,Consumer Durables,0.963703,0.451505,0.122099,50.841385,0.032709,0.044334,0.170843,0.059783,0.032497,0.075955,0.133060,0.993501,4.991711,3.094575,0.755774,0.214529,0.513185,9.693487,13.103448,4.094575,0.044334,10.491970,18.888889,0.074426,3.483510,2015,7
4,BBB,Whirlpool Corporation,WHR,Standard & Poor's Ratings Services,2016-10-24,Consumer Durables,0.957844,0.495432,0.141608,47.761126,0.042861,0.053770,0.177720,0.065354,0.046363,0.096945,0.186047,1.081710,5.437795,3.012780,0.750796,0.166966,0.451372,7.135348,14.257556,4.012780,0.053770,8.293505,15.808147,0.058065,3.857790,2016,7


In [12]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1632 entries, 218 to 1622
Data columns (total 33 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   Rating                              1632 non-null   object        
 1   Name                                1632 non-null   object        
 2   Symbol                              1632 non-null   object        
 3   Rating Agency Name                  1632 non-null   object        
 4   Date                                1632 non-null   datetime64[ns]
 5   Sector                              1632 non-null   object        
 6   currentRatio                        1632 non-null   float64       
 7   quickRatio                          1632 non-null   float64       
 8   cashRatio                           1632 non-null   float64       
 9   daysOfSalesOutstanding              1632 non-null   float64       
 10  netProfitMargin            

# Defining Target Variable

In [13]:
# Creating a target variable for model
def risk_default(Risk_Score):
    if Risk_Score >= 7:
        return 0
    else:
        return 1

In [14]:
data['Default_risk_indicator'] = data['Rating_score'].apply(risk_default)
data.head()

,Rating,Name,Symbol,Rating Agency Name,Date,Sector,currentRatio,quickRatio,cashRatio,daysOfSalesOutstanding,netProfitMargin,pretaxProfitMargin,grossProfitMargin,operatingProfitMargin,returnOnAssets,returnOnCapitalEmployed,returnOnEquity,assetTurnover,fixedAssetTurnover,debtEquityRatio,debtRatio,effectiveTaxRate,freeCashFlowOperatingCashFlowRatio,freeCashFlowPerShare,cashPerShare,companyEquityMultiplier,ebitPerRevenue,enterpriseValueMultiple,operatingCashFlowPerShare,operatingCashFlowSalesRatio,payablesTurnover,year,Rating_score,Default_risk_indicator
218,BB,Alcoa Corporation,AA,Egan-Jones Ratings Company,2015-10-14,Basic Industries,1.067388,0.441348,0.231697,16.426467,-0.077060,-0.041164,0.192874,-0.005983,-0.052580,-0.013634,-0.091400,0.682325,1.192652,0.518958,0.298544,-0.872017,0.553143,2.646699,3.045891,1.738297,-0.017055,4.875840,4.784838,0.078132,6.554750,2015,6,1
979,BB,"American Airlines Group, Inc.",AAL,Egan-Jones Ratings Company,2013-11-12,Transportation,1.037447,0.783065,0.082573,21.291553,-0.068579,-0.081517,1.000000,0.052313,-0.043380,-0.076566,0.671549,0.632551,1.388598,-16.480776,1.064596,0.158716,-3.613333,-14.958969,56.738589,-15.480776,-0.081517,8.751532,4.139936,0.025240,5.730263,2013,6,1
980,B,"American Airlines Group, Inc.",AAL,Moody's Investors Service,2015-06-30,Transportation,0.901526,0.675400,0.073986,138.984089,0.619652,0.690604,1.000000,0.913567,0.065843,0.105881,1.426027,0.106258,0.201482,20.658090,0.953828,0.102740,-0.724351,-3.109598,10.179021,21.658090,0.690604,9.322615,4.292946,0.662223,7.692084,2015,5,1
978,AA,Apple Inc.,AAPL,Standard & Poor's Ratings Services,2015-05-28,Technology,1.080113,0.670423,0.218194,34.863645,0.216144,0.292585,0.385880,0.287223,0.170420,0.317612,0.354200,0.788457,8.863218,1.078397,0.518860,0.261261,0.835664,8.199722,4.120730,2.078397,0.292585,10.936582,9.812225,0.326666,3.717645,2015,9,0
977,AA,Apple Inc.,AAPL,Standard & Poor's Ratings Services,2016-05-20,Technology,1.108771,0.725096,0.262002,26.313608,0.228458,0.310271,0.400599,0.304773,0.183814,0.345525,0.447355,0.804585,10.400739,1.433740,0.589110,0.263683,0.858637,12.128089,7.230655,2.433740,0.310271,7.845289,14.124814,0.347714,3.947281,2016,9,0


In [15]:
data.groupby('year')['Default_risk_indicator'].value_counts()

year  Default_risk_indicator
2005  0                           1
2009  1                           1
2010  0                           5
      1                           4
2011  0                          27
      1                          23
2012  0                         171
      1                         118
2013  0                         154
      1                         109
2014  0                         181
      1                         131
2015  0                         217
      1                         151
2016  1                         173
      0                         166
Name: count, dtype: int64

In [16]:
# Our scope of coverage for this project is from 2011 to 2016

data = data[data['year'] >= 2011]
data.sort_values(by=['Name', 'year'], inplace=True)

In [17]:
data.head()

,Rating,Name,Symbol,Rating Agency Name,Date,Sector,currentRatio,quickRatio,cashRatio,daysOfSalesOutstanding,netProfitMargin,pretaxProfitMargin,grossProfitMargin,operatingProfitMargin,returnOnAssets,returnOnCapitalEmployed,returnOnEquity,assetTurnover,fixedAssetTurnover,debtEquityRatio,debtRatio,effectiveTaxRate,freeCashFlowOperatingCashFlowRatio,freeCashFlowPerShare,cashPerShare,companyEquityMultiplier,ebitPerRevenue,enterpriseValueMultiple,operatingCashFlowPerShare,operatingCashFlowSalesRatio,payablesTurnover,year,Rating_score,Default_risk_indicator
1436,AA,3M Company,MMM,Moody's Investors Service,2012-06-15,Health Care,2.249586,1.387061,0.407829,47.666577,0.144642,0.203674,0.470028,0.208639,0.135469,0.230411,0.277756,0.936583,3.862640,1.050324,0.512272,0.277566,0.739023,5.511644,5.194072,2.050324,0.203674,8.743916,7.458010,0.178447,9.551430,2012,9,0
1435,AA,3M Company,MMM,Egan-Jones Ratings Company,2013-08-16,Health Care,1.698186,1.012270,0.344225,50.284895,0.150918,0.212562,0.478281,0.215931,0.138867,0.251881,0.266198,0.920149,3.568077,0.916924,0.478331,0.280555,0.713770,6.088869,4.893679,1.916924,0.212562,11.290941,8.530576,0.188429,8.952752,2013,9,0
1433,AA,3M Company,MMM,Egan-Jones Ratings Company,2015-01-28,Health Care,1.961487,1.127209,0.316272,48.611609,0.155746,0.220798,0.483140,0.224223,0.158496,0.278026,0.378061,1.017653,3.748498,1.385308,0.580767,0.288642,0.774676,7.906654,3.886322,2.385308,0.220798,12.911759,10.206408,0.208227,9.101826,2015,9,0
403,A,ABB Ltd,ABB,Moody's Investors Service,2012-06-15,Consumer Durables,1.436326,0.300405,0.290985,0.000000,0.087260,0.119768,0.300974,0.000237,0.083611,0.197081,0.210116,0.958182,7.718407,1.477594,0.587974,0.000000,0.986988,1.558129,2.174388,2.513025,0.119768,43.555179,1.578671,0.095078,5.545208,2012,8,0
404,A,ABB Ltd,ABB,Egan-Jones Ratings Company,2013-08-05,Consumer Durables,1.584708,0.372894,0.361079,0.000000,0.069466,0.097161,0.286561,0.104832,0.060482,0.129536,0.155638,0.870672,6.691398,1.544919,0.600366,0.000000,0.995073,1.582499,2.707009,2.573295,0.097161,9.225518,1.590335,0.087292,5.840376,2013,8,0


In [18]:
data.describe()

,Date,currentRatio,quickRatio,cashRatio,daysOfSalesOutstanding,netProfitMargin,pretaxProfitMargin,grossProfitMargin,operatingProfitMargin,returnOnAssets,returnOnCapitalEmployed,returnOnEquity,assetTurnover,fixedAssetTurnover,debtEquityRatio,debtRatio,effectiveTaxRate,freeCashFlowOperatingCashFlowRatio,freeCashFlowPerShare,cashPerShare,companyEquityMultiplier,ebitPerRevenue,enterpriseValueMultiple,operatingCashFlowPerShare,operatingCashFlowSalesRatio,payablesTurnover,year,Rating_score,Default_risk_indicator
count,1621,1621.000000,1621.000000,1621.000000,1621.000000,1621.000000,1621.000000,1621.000000,1621.000000,1621.000000,1621.000000,1621.000000,1.621000e+03,1.621000e+03,1621.000000,1621.000000,1621.000000,1621.000000,1.621000e+03,1.621000e+03,1621.000000,1621.000000,1621.000000,1.621000e+03,1621.000000,1621.000000,1621.000000,1621.000000,1621.000000
mean,2014-07-26 11:50:40.345465856,3.262115,2.916295,0.686375,387.744314,0.259049,0.410069,0.497952,0.573367,-46.971994,-92.613188,179.540321,4.603950e+03,9.097279e+03,1.731618,0.660758,0.143403,0.458071,1.119784e+03,7.748507e+02,2.727507,0.416696,45.059977,1.796332e+03,1.534727,33.807248,2014.033930,6.599013,0.434917
min,2011-01-25 00:00:00,-0.932005,-1.893266,-0.192736,-811.845623,-101.845815,-124.343612,-7.699871,-124.343612,-40213.178290,-87162.162160,-60.076923,-9.157477e+00,-2.679777e+01,-2556.419643,0.000000,-100.611015,-92.083333,-3.039290e+03,-1.915035e+01,-2555.419643,-124.343612,-3749.921337,-1.195049e+04,-4.461837,-76.662850,2011.000000,1.000000,0.000000
25%,2013-04-25 00:00:00,1.073432,0.612745,0.130630,23.114313,0.020456,0.026291,0.233011,0.044407,0.017238,0.028341,0.049234,3.850713e-01,1.037903e+00,1.034994,0.531434,0.150298,0.269616,4.398544e-01,1.527412e+00,2.036998,0.028352,6.117451,2.303377e+00,0.072878,2.226836,2013.000000,6.000000,0.000000
50%,2014-09-09 00:00:00,1.493296,0.987370,0.298305,42.285644,0.063817,0.083979,0.405592,0.106224,0.044849,0.073756,0.122346,7.018560e-01,3.826450e+00,1.660876,0.642090,0.300907,0.644130,2.120266e+00,3.713171e+00,2.662281,0.086997,9.250333,4.319305e+00,0.131882,5.759722,2014.000000,7.000000,0.000000
75%,2015-11-09 00:00:00,2.185772,1.446416,0.631235,59.519746,0.114342,0.144763,0.823634,0.175334,0.076574,0.135049,0.203683,1.094251e+00,8.475600e+00,2.672702,0.750796,0.371174,0.840246,4.168831e+00,8.007000e+00,3.689693,0.149260,12.924382,7.194295e+00,0.239411,9.527662,2015.000000,7.000000,1.000000
max,2016-12-23 00:00:00,1725.505005,1139.541703,125.917417,115961.637400,198.517873,309.694856,2.702533,410.182214,0.487826,2.439504,141350.211000,2.553149e+06,5.156884e+06,2561.871795,1.927839,67.777778,34.594086,1.745763e+06,1.101695e+06,2562.871795,309.694856,11153.607090,2.796610e+06,688.526591,20314.880400,2016.000000,10.000000,1.000000
std,NaN,43.203063,36.749698,3.941383,4954.504474,6.527346,9.599223,0.411495,12.070511,1304.615167,2629.305123,4929.625018,1.070038e+05,2.114219e+05,93.135259,0.206291,4.448395,2.943211,4.336425e+04,2.737094e+04,93.136376,9.598462,513.767714,6.946293e+04,21.069800,704.562727,1.488875,1.216491,0.495899


In [19]:
# We will be calculating the proxy values for LGD and EAD, since we do not have data for these values and we will be using them in our model to calculate the Expected Loss

# For LGD, we will be deriving values from debt ratio
data['LGD_proxy'] = data['debtRatio'].clip(0, 1)

# For EAD, we will be deriving values from Current Ratio
data['EAD_proxy'] = (1 / data['currentRatio']).clip(0, 1)


In [20]:
# Creating a rating movement over years as an additional feature

data['previous_rating'] = data.groupby('Symbol')['Rating_score'].shift(1).fillna(0)

data['Rating_downgrade'] = data['Rating_score'] < data['previous_rating']

data['Rating_downgrade'] = data['Rating_downgrade'].astype(int)

data.head()

,Rating,Name,Symbol,Rating Agency Name,Date,Sector,currentRatio,quickRatio,cashRatio,daysOfSalesOutstanding,netProfitMargin,pretaxProfitMargin,grossProfitMargin,operatingProfitMargin,returnOnAssets,returnOnCapitalEmployed,returnOnEquity,assetTurnover,fixedAssetTurnover,debtEquityRatio,debtRatio,effectiveTaxRate,freeCashFlowOperatingCashFlowRatio,freeCashFlowPerShare,cashPerShare,companyEquityMultiplier,ebitPerRevenue,enterpriseValueMultiple,operatingCashFlowPerShare,operatingCashFlowSalesRatio,payablesTurnover,year,Rating_score,Default_risk_indicator,LGD_proxy,EAD_proxy,previous_rating,Rating_downgrade
1436,AA,3M Company,MMM,Moody's Investors Service,2012-06-15,Health Care,2.249586,1.387061,0.407829,47.666577,0.144642,0.203674,0.470028,0.208639,0.135469,0.230411,0.277756,0.936583,3.862640,1.050324,0.512272,0.277566,0.739023,5.511644,5.194072,2.050324,0.203674,8.743916,7.458010,0.178447,9.551430,2012,9,0,0.512272,0.444526,0.0,0
1435,AA,3M Company,MMM,Egan-Jones Ratings Company,2013-08-16,Health Care,1.698186,1.012270,0.344225,50.284895,0.150918,0.212562,0.478281,0.215931,0.138867,0.251881,0.266198,0.920149,3.568077,0.916924,0.478331,0.280555,0.713770,6.088869,4.893679,1.916924,0.212562,11.290941,8.530576,0.188429,8.952752,2013,9,0,0.478331,0.588864,9.0,0
1433,AA,3M Company,MMM,Egan-Jones Ratings Company,2015-01-28,Health Care,1.961487,1.127209,0.316272,48.611609,0.155746,0.220798,0.483140,0.224223,0.158496,0.278026,0.378061,1.017653,3.748498,1.385308,0.580767,0.288642,0.774676,7.906654,3.886322,2.385308,0.220798,12.911759,10.206408,0.208227,9.101826,2015,9,0,0.580767,0.509817,9.0,0
403,A,ABB Ltd,ABB,Moody's Investors Service,2012-06-15,Consumer Durables,1.436326,0.300405,0.290985,0.000000,0.087260,0.119768,0.300974,0.000237,0.083611,0.197081,0.210116,0.958182,7.718407,1.477594,0.587974,0.000000,0.986988,1.558129,2.174388,2.513025,0.119768,43.555179,1.578671,0.095078,5.545208,2012,8,0,0.587974,0.696221,0.0,0
404,A,ABB Ltd,ABB,Egan-Jones Ratings Company,2013-08-05,Consumer Durables,1.584708,0.372894,0.361079,0.000000,0.069466,0.097161,0.286561,0.104832,0.060482,0.129536,0.155638,0.870672,6.691398,1.544919,0.600366,0.000000,0.995073,1.582499,2.707009,2.573295,0.097161,9.225518,1.590335,0.087292,5.840376,2013,8,0,0.600366,0.631031,8.0,0


In [21]:
# Over 154 companies have downgraded their rating compared to their last ratings as per dataset information

data['Rating_downgrade'].value_counts()

Rating_downgrade
0    1467
1     154
Name: count, dtype: int64

In [22]:
# The dataset is quite well balanced

data['Default_risk_indicator'].value_counts()

Default_risk_indicator
0    916
1    705
Name: count, dtype: int64

In [23]:
# Expoting to csv
data.to_csv("cleaned_data.csv", index=False)